In [2]:
import pandas as pd
import geopandas as gpd
import fiona
import numpy as np
import requests

This script starts from the list of fire we want to vaidate and sends of all the requests to the API

In [11]:
calfire_filtered_parks = gpd.read_file("Validation_Fire_Perimeters_2015_2024.shp")
calfire_filtered_parks.head()

,YEAR_,STATE,AGENCY,UNIT_ID,FIRE_NAME,INC_NUM,ALARM_DATE,CONT_DATE,CAUSE,C_METHOD,OBJECTIVE,GIS_ACRES,COMMENTS,COMPLEX_NA,IRWINID,FIRE_NUM,COMPLEX_ID,DECADES,geometry
0,2024,CA,NPS,KNP,COFFEE POT,00000088,2024/08/03 00:00:00,2024/12/16 00:00:00,1,3,1,14103.9000,None,None,{62A5DB78-38A8-4B54-8F7C-664DB0783E9C},None,None,2020-January 2025,"MULTIPOLYGON (((-13217941.346 4353257.821, -13..."
1,2024,CA,NPS,KNP,SENTINEL,00000058,2024/07/14 00:00:00,2024/11/08 00:00:00,14,7,1,260.6600,None,None,{5656FBF0-3D2E-4221-BA42-39E7723EC166},None,None,2020-January 2025,"POLYGON ((-13208263.579 4405096.209, -13208266..."
2,2024,CA,NPS,KNP,SIMPSON,00000074,2024/07/26 00:00:00,2024/09/02 00:00:00,14,2,1,52.1125,None,None,{65AA08F3-7325-4D42-8457-A6F0E12230AB},None,None,2020-January 2025,"POLYGON ((-13204355.841 4436805.636, -13204311..."
3,2023,CA,NPS,MNP,YORK,00010701,2023/07/28 00:00:00,2023/08/20 00:00:00,14,7,1,93077.9000,None,None,{B9E0F397-DE63-4B90-B5DA-9D04A7D2381B},None,None,2020-January 2025,"POLYGON ((-12815785.524 4224752.868, -12815785..."
4,2023,CA,NPS,KNP,REDWOOD,00000061,2023/08/15 00:00:00,2023/12/14 00:00:00,1,7,2,2248.4500,None,None,{405281C7-2B67-43C1-8F7A-F907AE53D92E},None,None,2020-January 2025,"POLYGON ((-13205607.284 4375004.509, -13205597..."


### Define functions

In [16]:

def create_bbox(fire_name, fires):
    ''' creates a buffered bounding box around a fire parameter '''

    fire = fires[fires['FIRE_NAME'] == fire_name].buffer(250).to_crs(epsg=4326)
    
    bbox = fire.bounds

    return bbox

def get_fire_date(fire_name, fires):
    ''' gets the alarm (start) date of a fire '''

    start_date = pd.to_datetime(fires[fires['FIRE_NAME'] == fire_name]['ALARM_DATE'].values[0]).to_datetime64()

    return start_date

def get_cont_date(fire_name, fires):
    ''' gets the containment date of a fire, returns None if missing '''

    raw = fires[fires['FIRE_NAME'] == fire_name]['CONT_DATE'].values[0]

    if pd.isnull(raw):
        return None

    return pd.to_datetime(raw).to_datetime64()

def create_query(fire_name, bbox, date_of_fire, post_fire_range, date_mode='alarm'):
    ''' Creates an API query for the bounding box and time period specified.
    
    date_mode: 'alarm' or 'cont' — recorded in the fire_event_name for downstream tracking.
    '''

    pre_start = date_of_fire - np.timedelta64(21, 'D')
    pre_end = date_of_fire - np.timedelta64(1, 'D')
    post_end = date_of_fire + np.timedelta64(post_fire_range, 'D')

    api_request = {
        "fire_event_name": f"{fire_name}_date{str(date_of_fire).split('T')[0]}_range{post_fire_range}_mode{date_mode}",
        "coarse_geojson": {
            "type": "Polygon",
            "coordinates": [[
                [float(bbox[0]), float(bbox[1])], 
                [float(bbox[2]), float(bbox[1])],
                [float(bbox[2]), float(bbox[3])], 
                [float(bbox[0]), float(bbox[3])], 
                [float(bbox[0]), float(bbox[1])]  
            ]]
        },
        "prefire_date_range": [str(pre_start).split('T')[0], str(pre_end).split('T')[0]],
        "postfire_date_range": [str(date_of_fire).split('T')[0], str(post_end).split('T')[0]]
    }
   
    return api_request


### Test if it works

In [17]:
url = "https://fire-recovery-backend-dev-113009620257.us-central1.run.app/fire-recovery/process/analyze_fire_severity" 

fire_name = 'YORK'
bbox = create_bbox(fire_name, calfire_filtered_parks)
date_of_fire = get_fire_date(fire_name, calfire_filtered_parks)
query_data = create_query(fire_name, bbox.values[0], date_of_fire, 15)
try:
    # Use the 'json' parameter: it automatically sets 'Content-Type: application/json'
    # and runs json.dumps() for you.
    response = requests.post(url, json=query_data)

    # 4. Check the results
    if response.status_code == 200 or response.status_code == 201:
        print("Success!")
        print(response.json()) # This is the data the API sends back
    else:
        print(f"Failed with status code: {response.status_code}")
        print(response.text) # This shows the error message from the API

except requests.exceptions.RequestException as e:
    print(f"A connection error occurred: {e}")

/opt/miniconda3/envs/burnseverity/lib/python3.13/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(


Success!
{'fire_event_name': 'YORK_date2023-07-28_range15_modealarm', 'status': 'Processing started', 'job_id': '8536bc78-ed44-4bc6-a1bd-22aa0b02dff7'}


In [18]:
# Get list of fire names to process
fire_names = calfire_filtered_parks['FIRE_NAME'].unique()

# Define post-fire day ranges to test
post_fire_days = [5, 10, 15, 21, 30, 45, 60, 90]

# date_modes: 'alarm' uses ALARM_DATE as the reference, 'cont' uses CONT_DATE
date_modes = ['alarm', 'cont']

# Initialize list to collect results
results = []

url = "https://fire-recovery-backend-dev-113009620257.us-central1.run.app/fire-recovery/process/analyze_fire_severity"

# Loop through each fire, date mode, and post-fire day range
for fire_name in fire_names:
    bbox = create_bbox(fire_name, calfire_filtered_parks)

    for date_mode in date_modes:
        if date_mode == 'alarm':
            date_of_fire = get_fire_date(fire_name, calfire_filtered_parks)
        else:
            date_of_fire = get_cont_date(fire_name, calfire_filtered_parks)

        if date_of_fire is None:
            for post_fire_range in post_fire_days:
                results.append({
                    'fire_event_name': None,
                    'job_id': None,
                    'fire_name': fire_name,
                    'date_mode': date_mode,
                    'post_fire_days': post_fire_range,
                    'status': 'skipped_no_cont_date'
                })
            print(f"- {fire_name} ({date_mode}): skipped — no CONT_DATE")
            continue

        for post_fire_range in post_fire_days:
            query_data = create_query(fire_name, bbox.values[0], date_of_fire, post_fire_range, date_mode)
            
            try:
                response = requests.post(url, json=query_data)
                
                if response.status_code == 200 or response.status_code == 201:
                    response_data = response.json()
                    results.append({
                        'fire_event_name': response_data.get('fire_event_name'),
                        'job_id': response_data.get('job_id'),
                        'fire_name': fire_name,
                        'date_mode': date_mode,
                        'post_fire_days': post_fire_range,
                        'status': 'success'
                    })
                    print(f"✓ {fire_name} ({date_mode}, {post_fire_range} days): {response_data.get('job_id')}")
                else:
                    results.append({
                        'fire_event_name': None,
                        'job_id': None,
                        'fire_name': fire_name,
                        'date_mode': date_mode,
                        'post_fire_days': post_fire_range,
                        'status': f'failed_{response.status_code}'
                    })
                    print(f"✗ {fire_name} ({date_mode}, {post_fire_range} days): Failed with {response.status_code}")
                    
            except requests.exceptions.RequestException as e:
                results.append({
                    'fire_event_name': None,
                    'job_id': None,
                    'fire_name': fire_name,
                    'date_mode': date_mode,
                    'post_fire_days': post_fire_range,
                    'status': 'error'
                })
                print(f"✗ {fire_name} ({date_mode}, {post_fire_range} days): Connection error")

# Create dataframe and save to CSV
df_results = pd.DataFrame(results)
df_results.to_csv('fire_processing_jobs.csv', index=False)

print(f"\nProcessed {len(results)} requests. Results saved to fire_processing_jobs.csv")
display(df_results)


/opt/miniconda3/envs/burnseverity/lib/python3.13/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(


✓ COFFEE POT (alarm, 5 days): 674b9f1e-0dd5-43b6-9717-998a5c8b10e9
✓ COFFEE POT (alarm, 10 days): 36dce962-a982-4e60-af53-eb061e0c6428
✓ COFFEE POT (alarm, 15 days): 7baa9569-e731-4d08-8869-7d2fcdbac731
✓ COFFEE POT (alarm, 21 days): 9b02f9a0-3373-4a8c-8743-1c58ab2f8b5c
✓ COFFEE POT (alarm, 30 days): eadd650b-94ba-4088-8b3f-9a3666ff4e1b
✓ COFFEE POT (alarm, 45 days): ddec762b-de4d-4670-a73d-12a53ac300da
✓ COFFEE POT (alarm, 60 days): 85002f05-85cf-47f7-8107-1c9173263016
✓ COFFEE POT (alarm, 90 days): 9c90c82e-1e02-443c-86c3-6fb44eedacf4
✓ COFFEE POT (cont, 5 days): f6c77f6a-c251-4b58-9ff7-bd0ab2791b61
✓ COFFEE POT (cont, 10 days): 2e73c62b-1cae-406f-931d-57a63dcd288e
✓ COFFEE POT (cont, 15 days): 8e1c2515-8071-415d-b9f6-541719c209b6
✓ COFFEE POT (cont, 21 days): 3eece749-1f67-4008-b030-80c3dedcf79b
✓ COFFEE POT (cont, 30 days): 45f60628-3113-485a-bc9d-df3783113ecd
✓ COFFEE POT (cont, 45 days): cbae0be9-978a-4f26-a4bf-3d069b19679e
✓ COFFEE POT (cont, 60 days): 5dd1f78f-713e-461f-96d2-90

/opt/miniconda3/envs/burnseverity/lib/python3.13/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(


✓ SENTINEL (alarm, 5 days): 6188fe33-d50d-41f5-9d08-2d446945bacd
✓ SENTINEL (alarm, 10 days): a573b98a-eaa7-421f-96cb-4b8096b04c6f
✓ SENTINEL (alarm, 15 days): 028587b6-754d-4215-9588-d1cf6706b1ac
✓ SENTINEL (alarm, 21 days): 9850a737-5679-43d6-bd2e-7803191c0951
✓ SENTINEL (alarm, 30 days): cf5b6bf5-34be-4204-af7b-6078f68af6e2
✓ SENTINEL (alarm, 45 days): b15b6e91-e846-4b3c-871a-9db35dfbf6ab
✓ SENTINEL (alarm, 60 days): bd468e18-c224-497e-a64e-d1a9da65e444
✓ SENTINEL (alarm, 90 days): cc96b483-f3d4-4695-a34c-9813cca9a1bd
✓ SENTINEL (cont, 5 days): c0055acf-fe07-4550-befc-b805ba553cff
✓ SENTINEL (cont, 10 days): a0f23e7b-271f-4f4d-ac36-9d3fede47fef
✓ SENTINEL (cont, 15 days): 303b11a7-cb15-42a3-9a0a-f257e84e2509
✓ SENTINEL (cont, 21 days): a78f6cb0-390f-4d0d-8eb5-c533e2ccc866
✓ SENTINEL (cont, 30 days): b143a40e-0392-4f10-9397-2336f776a582
✓ SENTINEL (cont, 45 days): 3755b83e-726a-41f2-bf55-1e8c3594734f
✓ SENTINEL (cont, 60 days): 5ddcc59e-0615-4c4a-ab9b-534f7d6d62bd
✓ SENTINEL (cont, 9

/opt/miniconda3/envs/burnseverity/lib/python3.13/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(


...

/opt/miniconda3/envs/burnseverity/lib/python3.13/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(


✓ MULHOLLAND (alarm, 5 days): 3091560d-af36-4421-9b62-be0e86e16b05
✓ MULHOLLAND (alarm, 10 days): 353ae86f-3a60-4036-b06d-dde0d177bf3a
✓ MULHOLLAND (alarm, 15 days): 7b1c471c-8c25-4254-88c4-7460eaf7554c
✓ MULHOLLAND (alarm, 21 days): 36cf51c6-d780-4d34-ad6b-5f42c11a4327
✓ MULHOLLAND (alarm, 30 days): 721925b2-6cc2-4994-a5ac-1d65cf333b9a
✓ MULHOLLAND (alarm, 45 days): 485585c7-fac1-4972-b0f3-998e838e86b7
✓ MULHOLLAND (alarm, 60 days): 0ee8c9b9-fd08-441b-9fd3-ec693c6d842a
✓ MULHOLLAND (alarm, 90 days): b82cd269-90b9-42fd-a34e-15707a7f5510
✓ MULHOLLAND (cont, 5 days): d086dddb-d19a-47b7-bde0-bb22b371e4f3
✓ MULHOLLAND (cont, 10 days): feb59c36-1302-4b5a-a53f-7d257f69769f
✓ MULHOLLAND (cont, 15 days): 6bf6b36c-6d29-49bd-9662-65ce1775a777
✓ MULHOLLAND (cont, 21 days): eba0189b-3ba2-4d41-9fa6-f6e42b3c3ec8
✓ MULHOLLAND (cont, 30 days): 670a6e4d-aecf-4ab4-8375-796627d0da7a
✓ MULHOLLAND (cont, 45 days): 84c7a72b-084a-4e5f-bae4-ff7d89a3988d
✓ MULHOLLAND (cont, 60 days): 3c2068e6-abae-4e28-bb6a-bd

/opt/miniconda3/envs/burnseverity/lib/python3.13/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(


✓ COAST (alarm, 5 days): a7d4ad31-abe6-44c1-aaee-ca883e6ef01c
✓ COAST (alarm, 10 days): 637c872a-19df-42cb-bdf0-d84a68f88836
✓ COAST (alarm, 15 days): 957b18e1-10bf-47ff-80a0-e1fa39247fe7
✓ COAST (alarm, 21 days): 6c38d611-b053-48f2-bb68-4e01ca7ba43c
✓ COAST (alarm, 30 days): 584a005e-307c-4db9-9358-b5eee797a024
✓ COAST (alarm, 45 days): f6e71754-75c4-451c-9340-e40d4c643599
✓ COAST (alarm, 60 days): 63ce099c-0719-4310-bc65-645ed9ece572
✓ COAST (alarm, 90 days): 02d5d595-ab59-4e67-ba38-32aba392ee24
✓ COAST (cont, 5 days): 31d9c9fa-7b8b-4b97-ad50-db7b109d07f0
✓ COAST (cont, 10 days): cb095b7c-8a23-44c6-808c-1b243373479c
✓ COAST (cont, 15 days): 568a9f54-169d-4f22-a5f2-312e2227df1d
✓ COAST (cont, 21 days): fa62c39a-c860-45db-8b6e-203a7e1e14f6
✓ COAST (cont, 30 days): f9d375f6-8b1e-4638-ae19-d64d0dca1a24
✓ COAST (cont, 45 days): 619ad160-abc5-4419-8b0c-e8cf596d0590
✓ COAST (cont, 60 days): 7da5090a-3b25-48ba-81f8-894c0e83b070
✓ COAST (cont, 90 days): 4c45c0f2-ee87-478f-b167-b616495fc7b2
✓ 

,fire_event_name,job_id,fire_name,date_mode,post_fire_days,status
0,COFFEE POT_date2024-08-03_range5_modealarm,674b9f1e-0dd5-43b6-9717-998a5c8b10e9,COFFEE POT,alarm,5,success
1,COFFEE POT_date2024-08-03_range10_modealarm,36dce962-a982-4e60-af53-eb061e0c6428,COFFEE POT,alarm,10,success
2,COFFEE POT_date2024-08-03_range15_modealarm,7baa9569-e731-4d08-8869-7d2fcdbac731,COFFEE POT,alarm,15,success
3,COFFEE POT_date2024-08-03_range21_modealarm,9b02f9a0-3373-4a8c-8743-1c58ab2f8b5c,COFFEE POT,alarm,21,success
4,COFFEE POT_date2024-08-03_range30_modealarm,eadd650b-94ba-4088-8b3f-9a3666ff4e1b,COFFEE POT,alarm,30,success
...,...,...,...,...,...,...
651,LIBERTY CANYON_date2016-11-05_range21_modecont,b39bdd79-9245-4cc9-9828-e0704dc0f088,LIBERTY CANYON,cont,21,success
652,LIBERTY CANYON_date2016-11-05_range30_modecont,90bcb475-2340-4554-ad80-b75f7f142799,LIBERTY CANYON,cont,30,success
653,LIBERTY CANYON_date2016-11-05_range45_modecont,e83fb764-852c-4a2a-8847-ca82863c1918,LIBERTY CANYON,cont,45,success
654,LIBERTY CANYON_date2016-11-05_range60_modecont,edcafa48-8fb2-4a37-b50b-e1ab2ee0e01b,LIBERTY CANYON,cont,60,success
